# DBMS Lecture 1: Python Connectivity & Database Design Principles

## Table of Contents
1. [Introduction to Jupyter Notebooks](#introduction-to-jupyter-notebooks)
2. [Prerequisites and Environment Setup](#prerequisites-and-environment-setup)
   - [Python Installation](#python-installation)
   - [Virtual Environment Configuration](#virtual-environment-configuration-pre-notebook-step)
3. [Topic 1: Python-MySQL Connectivity](#topic-1-python-mysql-connectivity)
   - [Client-Server Architecture and Driver Purpose](#client-server-architecture-and-driver-purpose)
   - [Establishing Connections & Securing Credentials](#establishing-connections-securing-credentials)
   - [Transaction Management](#transaction-management-atomicity-and-durability)
4. [Hands-On Lab 1: Personal Assignment Tracker](#hands-on-lab-1-personal-assignment-tracker)
5. [Topic 2: Database Design Principles: Normalization and De-Normalization](#topic-2-database-design-principles-normalization-and-de-normalization)
   - [Understanding Flat Files](#understanding-flat-files)
   - [Identification of Database Anomalies](#identification-of-database-anomalies)
   - [Database Keys Primer](#database-keys-primer)
   - [The Normal Forms Hierarchy](#the-normal-forms-hierarchy)
   - [Normalization vs. De-Normalization Comparison](#normalization-vs-de-normalization-comparison)
6. [Hands-On Lab 2: University Course Manager](#hands-on-lab-2-university-course-manager)


## Introduction to Jupyter Notebooks

Jupyter Notebooks are interactive web-based environments that allow you to combine executable code, rich text (Markdown), and visual output in a single document.

### Key Concepts

1.  **Cells**: The workspace is divided into discrete "cells".
    -   **Markdown Cells**: Utilized for documentation, formulas, and explanations.
    -   **Code Cells**: Utilized for writing and executing code (often Python).
2.  **Execution Order**: The order in which you execute cells matters! State (variables, functions) is maintained in memory across the entire notebook.
3.  **Kernel**: The backend engine that executes the code. If a Code Cell hangs, the Kernel can be restarted.

### Essential Navigation

-   **Execute a Cell**: Press `Shift + Enter` or click the **Run** button ($\blacktriangleright$) in the toolbar.
-   **Edit a Markdown Cell**: Double-click it to see the raw text. Run it to render it back to rich text.
-   **Package Installation (Magics)**: Use `%pip install <package>` to install packages in the *current* active kernel (preferred over `!`).
-   **System Shell Access**: Use `!` to run operating system commands (e.g., `!ls` on Linux/macOS or `!dir` on Windows).


## Prerequisites and Environment Setup

The execution of the laboratory exercises requires the preparation of a suitable Python environment. The utilization of a Virtual Environment is recommended to maintain isolation from system-wide packages.

### Python Installation

The installation of Python (version 3.10 or higher) is recommended, as older versions (such as 3.8 and 3.9) have reached End-of-Life (EOL) status.
-   [Python Official Website](https://www.python.org/downloads/)

### MySQL Server Instance Prerequisite

Prior to executing programmatic connection tests, a MySQL Server instance must be operational and accessible.

-   **Locality**: The server typically runs on standard port `3306` at `127.0.0.1` (localhost).
-   **Access Credentials**: 
    -   *Local Installations*: The `user` is typically `root`, and the password is specified during the initial setup wizard.
    -   *Docker Containers*: Credentials are set in the `docker-compose.yml` file or via `-e MYSQL_ROOT_PASSWORD` flags.
    -   *University Lab*: Consult your system administrator for provisioned credentials.

> [!NOTE]
> **Network Addresses: 127.0.0.1 vs. 0.0.0.0**
> -   `127.0.0.1` (localhost): A loopback address. It means "this machine" and is only accessible internally.
> -   `0.0.0.0`: A non-routable meta-address utilized by **servers** to bind to *all* available network interfaces (WiFi, Ethernet). If a server binds to `0.0.0.0`, it can accept connections from external machines.
> -   **Rule of Thumb**: Servers listen on `0.0.0.0` (to allow access); Clients connect to `127.0.0.1` (or the specific IP of the server).

### Database GUI Client Installation (DBeaver / MySQL Workbench)

While database operations can be performed programmatically via Python, the utilization of a Graphical User Interface (GUI) is recommended for visual schema verification and manual query execution.

#### 1. DBeaver Community Edition (Recommended / Preferred)
DBeaver is a universal, open-source database manager. It is highly flexible, actively maintained, and works seamlessly across Windows, Mac, and Linux.
-   **Official Download**: [DBeaver Official Site](https://dbeaver.io/download/)

#### 2. MySQL Workbench 8.0 (Alternative)
A standard client developed by MySQL. 
-   **Official Download**: [MySQL Workbench Downloads](https://dev.mysql.com/downloads/workbench/)

#### Linux (Debian/Ubuntu) Package Installation
If using Linux, MySQL Workbench can often be installed via the system package manager (though availability varies by distribution version):
```bash
sudo apt update
sudo apt install mysql-workbench
```
DBeaver is typically installed via a `.deb` package downloaded from the official site or via Flatpak/Software Center.


### Virtual Environment Configuration (Pre-Notebook Step)

The creation and activation of a virtual environment should be performed in the system terminal **prior** to launching the Jupyter server.

`venv` is the official Python standard for virtual environments. It is included by default on Windows and macOS, but many Linux distributions (e.g., Ubuntu/Debian) require a separate installation of the `python3-venv` package.

-   [Python venv Documentation](https://docs.python.org/3/library/venv.html)

#### 1. (Linux Only) Install venv module
```bash
sudo apt install python3-venv
```

#### 2. Create Virtual Environment
```bash
# macOS/Linux
python3 -m venv venv_dbms

# Windows
python -m venv venv_dbms
```

#### 3. Activate Virtual Environment
```bash
# macOS/Linux
source venv_dbms/bin/activate

# Windows
venv_dbms\Scripts\activate
```

#### 4. Register Kernel with Jupyter
To utilize the virtual environment within this notebook, the `ipykernel` package must be installed and registered.

```bash
# Install ipykernel
pip install ipykernel

# Register the kernel
python -m ipykernel install --user --name=venv_dbms --display-name="Python (DBMS Venv)"
```


### MySQL Driver Installation

The `mysql-connector-python` package facilitates programmatic communication with MySQL. Running the following cell installs the driver in the active kernel.


In [ ]:
# Driver Installation
# Utilizing %pip ensures that the installation is bound to the active Jupyter kernel
%pip install mysql-connector-python

# Topic 1: Python-MySQL Connectivity

This section explores the methodologies for establishing programmatic communication between a Python application and a MySQL Database Management System.

### Client-Server Architecture and Driver Purpose

A relational database typically operates on a client-server architecture model:
-   **The Server**: Responsible for data persistence, transaction management, and query execution (e.g., MySQL Server).
-   **The Client**: The application requesting data or requesting modifications to data (e.g., Python script or GUI tool like MySQL Workbench 8.0).

The **Database Driver** (e.g., `mysql-connector-python`) serves as the intermediary protocol translator. It translates Pythonic calls into network packets that the MySQL server can interpret.

### GUI vs. Programmatic Clients

Database interactions can be categorized by the interface utilized:

1.  **GUI Clients (e.g., MySQL Workbench 8.0)**: Visual interfaces optimized for human interaction, schema design, and administration.
2.  **Programmatic Clients (e.g., Python scripts using drivers)**: Automated interfaces optimized for application logic.

### Establishing Connections & Securing Credentials

Hardcoding credentials within source code is a significant security vulnerability. Utilization of environment variables or external configuration files is considered an industry best practice.

#### 1. Setting Environment Variables in Jupyter
In a Jupyter Notebook runner, environment variables can be set using standard notebook magic:
```python
%env MYSQL_DB_PASSWORD=your_secure_password
```

#### 2. Local Development (.env files & python-dotenv)
For standard Python scripts, credentials are often stored in a `.env` file (added to `.gitignore`) and loaded using `python-dotenv`.

-   **Location**: By default, `load_dotenv()` searches the **Current Working Directory (CWD)** and its parent directories for a `.env` file. It is standard practice to place the `.env` file in the **project root directory**.
-   **Explicit Path**: If the file is placed elsewhere, specify the path: `load_dotenv(dotenv_path="path/to/.env")`.

**Example `.env` file:**
```text
MYSQL_DB_PASSWORD=YOUR_PASSWORD_HERE
```

**Example Python Usage:**
```python
# %pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv() # Searches CWD and parents for .env
db_password = os.environ.get('MYSQL_DB_PASSWORD')
```

#### Secrets Management in Production Systems
In production software engineering layouts, secrets are managed via specialized infrastructure:

-   **Container Orchestration (e.g., Kubernetes Secrets)**: Secrets are injected into container environment variables at runtime.
-   **Secret Management Services (e.g., Google Cloud Secret Manager, HashiCorp Vault)**: Applications authenticate with the service and retrieve secrets programmatically via secure APIs.


In [ ]:
# System Prerequisite: Programmatic Database Creation
import os
import mysql.connector

# To create a database, we connect to the MySQL Server without specifying a target database name first.
try:
    connection = mysql.connector.connect(
        host="127.0.0.1",
        user="student",
        password=os.environ.get("MYSQL_DB_PASSWORD", "dummy_password"),
    )
    cursor = connection.cursor()

    # DDL: Create Database if it does not exist
    cursor.execute("CREATE DATABASE IF NOT EXISTS student_tasks")
    print("Database 'student_tasks' verified/created successfully.")

    cursor.close()
    connection.close()
except mysql.connector.Error as err:
    print(f"Failed to verify or create database: {err}")
    # TROUBLESHOOTING: Error 1044 (Access denied) occurs if the 'student' user lacks permissions to create NEW databases.
    #
    # Why this happens: If the user was granted permissions only on 'student_tasks.*', they cannot create 'student_tasks_v2'.
    #
    # Resolution (Run as root in terminal):
    #   GRANT ALL PRIVILEGES ON student_tasks_v2.* TO 'student'@'localhost';
    #   FLUSH PRIVILEGES;
    #
    # (Or use 'student_%' to allow the student to create any database starting with 'student_')

In [ ]:
import os
import mysql.connector
from mysql.connector import errorcode

# Setting the environment variable inside the notebook for demonstration.
# Pressing Shift+Enter runs this magic command to set the variable in the active session.
%env MYSQL_DB_PASSWORD=dummy_password


def establish_connection():
    """
    Establishes a connection to the MySQL server using environment variables.
    Industry Practice: Read credentials from environment to prevent hardcoding.
    """
    config = {
        "user": "student",
        "password": os.environ.get("MYSQL_DB_PASSWORD", "dummy_password"),
        "host": "127.0.0.1",
        "database": "student_tasks",
    }

    try:
        # **config: Dictionary Unpacking (Kwargs Expansion)
        # This expands the 'config' dictionary into keyword arguments expected by the connect() function.
        # Conceptually equivalent to: connect(user='root', password='...', host='...', database='...')
        connection = mysql.connector.connect(**config)
        print("Connection successfully established.")
        return connection
    except mysql.connector.Error as err:
        # Exception Handling: Robust systems intercept specific error codes to provide actionable feedback.
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Access denied: Invalid username or password.")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist.")
        else:
            print(f"Connection failed: {err}")
        return None


connection = establish_connection()

### Utilizing Database Cursors

While a **Connection** establishes the communication pipeline to the MySQL server, a **Cursor** is the object that executes queries and retrieves results.

#### The Telephone Analogy
-   **Connection**: The active communication established between the client application and the MySQL Server.
-   **Cursor**: The execution agent at the server (the internal engine) that receives the SQL instruction, processes it, and returns the results to the client row-by-row.

A single Connection context can be utilized to generate multiple Cursors for distinct queries.

-   *Industry Practice*: Cursors consume memory on the database server. Explicit closure (`cursor.close()`) in a `finally` block is required to release resources and prevent connection leakage.

### Secure Query Execution (SQL Injection Prevention)

The dynamic construction of SQL queries using string concatenation or f-strings is a critical security vulnerability known as **SQL Injection (SQLi)**.

#### Why String Interpolation is Vulnerable
When utilizing f-strings or concatenation (`+`), Python merges the data directly into the SQL instruction string before sending it to the database. The database server cannot distinguish between the intended command structure and the data.

-   **Intended Query**: `SELECT * FROM tasks WHERE id = {user_input}`
-   **Malicious Input**: If the input is `'1 OR 1=1'`, the final query becomes `SELECT * FROM tasks WHERE id = 1 OR 1=1`. 
-   **Result**: The condition `1=1` is always true, which might leak all records or override authentication.

#### Why Parameterization (Safe Approach) Works
When utilizing placeholders (typically `%s`), the query structure and data are separated. The database engine parses the query structure first, and then binds the data strictly as data (literal values), never as executable code.

-   **Parameterized Query**: `SELECT * FROM tasks WHERE id = %s` with input `'1 OR 1=1'`.
-   **Result**: The database searches for a task whose ID is *literally* the text `'1 OR 1=1'`, which fails harmlessly.

**Vulnerable Comparison (Anti-pattern):**
```python
# Danger: The variable is interpolated before sending to DBMS
query = f"SELECT * FROM tasks WHERE id = {user_id}"
cursor.execute(query)
```

**Secure Comparison (Standard Practice):**
```python
# Safe: The structure is sent, then the data is bound
query = "SELECT * FROM tasks WHERE id = %s"
cursor.execute(query, (user_id,))
```


### Database and Table Hierarchy Mapping

In relational database systems, tables do not exist in isolation; they are scoped within a specific Database container.

-   **Active Database Container**: `student_tasks`
-   **Contained Table (Entity)**: `assignments`

Before executing Data Definition Language (DDL) or Data Manipulation Language (DML), the connection must point to the correct database context.

---

### Schema Prerequisite (DDL Before DML)

Before executing Data Manipulation Language (DML) queries such as `INSERT`, `UPDATE`, or `DELETE`, the target table must exist.

#### Tabular Schema Definition (`assignments` Table)

| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `task_id` | `INT` | `PRIMARY KEY`, `AUTO_INCREMENT` | Unique identifier for the assignment record automatically generated by the database. |
| `title` | `VARCHAR(255)` | `NOT NULL` | The title or description of the task (Mandatory). |
| `subject` | `VARCHAR(100)` | - | The academic subject or category. |
| `due_date` | `DATE` | - | The deadline for the task. |
| `is_completed` | `BOOLEAN` | `DEFAULT FALSE` | Status flag indicating whether the task is completed. |

#### Standard SQL DDL Syntax Example:
```sql
CREATE TABLE IF NOT EXISTS assignments (
    task_id INT AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    subject VARCHAR(100),
    due_date DATE,
    is_completed BOOLEAN DEFAULT FALSE
);
```

The laboratory setup functions automatically initialize the required tables programmatically via Python.

---

### Transaction Management (Atomicity and Durability)

When performing modifications to data (Insert, Update, Delete), changes are not immediately permanent. They are managed within a context called a **Transaction**.

#### The "Online Shopping Cart" Analogy
-   **Executing a query** is like **adding an item to your shopping cart**. It is reserved for your session, but the purchase isn't finalized yet.
-   **`connection.commit()` (Checkout / Confirm Purchase)**: This finalizes the transaction. The data is written to permanent storage. Even if the server crashes a millisecond later, the purchase is recorded (**Durability**).
-   **`connection.rollback()` (Empty Cart / Cancel Order)**: If an error occurs (e.g., payment fails), the order is canceled. The cart is emptied, and no partial or corrupt order is processed—reverting back to how things were before you started (**Atomicity**).

In Python database drivers, transactions are on by default. Modified data must be explicitly `.commit()`-ed to persist, or `.rollback()`-ed to discard upon encountering an exception.


In [ ]:
# Table Initialization: Establishing connection and running DDL.
connection = establish_connection()

if connection and connection.is_connected():
    cursor = connection.cursor()

    create_table_query = """
    CREATE TABLE IF NOT EXISTS assignments (
        assignment_id INT AUTO_INCREMENT PRIMARY KEY,
        title VARCHAR(255) NOT NULL,
        subject VARCHAR(100),
        due_date DATE,
        is_completed BOOLEAN DEFAULT FALSE
    )
    """

    try:
        cursor.execute(create_table_query)
        connection.commit()
        print("Table 'assignments' initialized successfully.")
    except mysql.connector.Error as err:
        print(f"Failed to initialize table: {err}")
    finally:
        cursor.close()
        connection.close()

In [ ]:
def add_assignment(connection, title, subject, due_date):
    """
    Inserts a new assignment into the assignments table using parameterized queries.
    """
    # Parameterized query (%s as placeholders) prevents SQL Injection.
    query = """
    INSERT INTO assignments (title, subject, due_date, is_completed)
    VALUES (%s, %s, %s, %s)
    """
    cursor = connection.cursor()
    try:
        cursor.execute(query, (title, subject, due_date, False))
        connection.commit()  # Industry Practice: Explicitly commit transactions to ensure durability.
        print("Assignment added successfully.")
    except mysql.connector.Error as err:
        connection.rollback()  # Industry Practice: Rollback on failure to prevent partial/corrupt state.
        print(f"Failed to add assignment: {err}")
    finally:
        # The 'finally' block executes regardless of whether an exception occurred.
        # It is the standard idiom for resource cleanup (closing cursors/connections).
        cursor.close()


def view_assignments(connection):
    """
    Retrieves and displays all assignments.
    """
    query = (
        "SELECT assignment_id, title, subject, due_date, is_completed FROM assignments"
    )
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        records = cursor.fetchall()  # fetchall() pulls all resulting rows into memory.
        print("\n--- Assignments ---")
        for row in records:
            # Inline conditional (ternary operator) for concise boolean evaluation.
            status = "Completed" if row[4] else "Pending"
            print(
                f"ID: {row[0]} | Title: {row[1]} | Subject: {row[2]} | Due: {row[3]} | Status: {status}"
            )
    except mysql.connector.Error as err:
        print(f"Failed to retrieve assignments: {err}")
    finally:
        cursor.close()

In [ ]:
# Execution Caller: Populating multiple sample records to demonstrate sorting and viewing.
connection = establish_connection()

if connection and connection.is_connected():
    # Adding various records with different subjects and due dates to make sorting meaningful.
    add_assignment(
        connection,
        title="Fix Workbench Connection Issue",
        subject="Prerequisites",
        due_date="2026-04-04",
    )
    add_assignment(
        connection,
        title="Read Normalization Chapter",
        subject="DBMS",
        due_date="2026-04-05",
    )
    add_assignment(
        connection,
        title="Practice Python Sequences",
        subject="Python",
        due_date="2026-04-07",
    )
    add_assignment(
        connection,
        title="Group Discussion: ACID Properties",
        subject="DBMS",
        due_date="2026-04-08",
    )
    add_assignment(
        connection,
        title="Submit DBMS Assignment 1",
        subject="DBMS",
        due_date="2026-04-10",
    )

    print("\n--- Current Assignments Table State ---")
    view_assignments(connection)

    connection.close()

In [ ]:
def complete_assignment(connection, assignment_id):
    """
    Updates an assignment to toggle the completion status.
    """
    query = "UPDATE assignments SET is_completed = %s WHERE assignment_id = %s"
    cursor = connection.cursor()
    try:
        cursor.execute(query, (True, assignment_id))
        connection.commit()
        print(f"Assignment ID {assignment_id} marked as completed.")
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Failed to complete assignment: {err}")
    finally:
        cursor.close()


def remove_assignment(connection, assignment_id):
    """
    Deletes an assignment from the database.
    """
    query = "DELETE FROM assignments WHERE assignment_id = %s"
    cursor = connection.cursor()
    try:
        # Tuple Requirement: (assignment_id,)
        # The trailing comma is required by Python to syntax-differentiate a single-element tuple from a scalar bound by parentheses.
        cursor.execute(query, (assignment_id,))
        connection.commit()
        print(f"Assignment ID {assignment_id} removed successfully.")
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Failed to remove assignment: {err}")
    finally:
        cursor.close()

In [ ]:
# Execution Caller: Completing an assignment and Verifying
connection = establish_connection()

if connection and connection.is_connected():
    # Assignment ID is assumed to be 1 (auto-incremented from previous insertions)
    complete_assignment(connection, assignment_id=1)

    # Verify the update
    view_assignments(connection)

    connection.close()

### Sorting and Aggregating Data (ORDER BY and GROUP BY)

To enhance the capabilities of the CLI application, sorting and grouping operations can be utilized.

#### Sorting (ORDER BY)
To retrieve records sorted by a specific column (e.g., chronologically by due date), the `ORDER BY` clause is applied.

#### Grouping (GROUP BY)
To perform aggregations (e.g., counting the number of assignments per subject), the `GROUP BY` clause is used in conjunction with aggregate functions like `COUNT()`.

-   *Key Concept for Evaluation*: Sorting large datasets requires computational resources. Utilizing the database engine for sorting (via `ORDER BY`) is generally more efficient than pulling all data into application memory (Python) and sorting it there, as database engines are specialized for these operations.


In [ ]:
def view_assignments_sorted(connection):
    """
    Retrieves assignments sorted by due date.
    """
    query = "SELECT assignment_id, title, subject, due_date FROM assignments ORDER BY due_date ASC"
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        records = cursor.fetchall()
        print("\n--- Assignments Sorted by Due Date ---")
        for row in records:
            print(f"ID: {row[0]} | Title: {row[1]} | Subject: {row[2]} | Due: {row[3]}")
    except mysql.connector.Error as err:
        print(f"Failed to fetch sorted assignments: {err}")
    finally:
        cursor.close()


def view_assignment_summary(connection):
    """
    Counts assignments per subject using GROUP BY.
    """
    query = "SELECT subject, COUNT(*) FROM assignments GROUP BY subject"
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        records = cursor.fetchall()
        print("\n--- Assignment Summary by Subject ---")
        for row in records:
            print(f"Subject: {row[0]} | Total Assignments: {row[1]}")
    except mysql.connector.Error as err:
        print(f"Failed to fetch assignment summary: {err}")
    finally:
        cursor.close()

In [ ]:
# Execution Caller: View Assignments Sorted by Due Date
connection = establish_connection()

if connection and connection.is_connected():
    view_assignments_sorted(connection)
    connection.close()

### Database Cleanup Operations (DELETE vs. TRUNCATE vs. DROP)

When removing data or structures from a database, three distinct SQL commands are utilized depending on the scope, speed requirements, and safety (transaction logging).

#### Comparison Table

| Command | Category | Operation Scope | Transaction Logging (Undo-able) | Speed | Resets Auto-Increment |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`DELETE`** | DML | Specific Rows (can use `WHERE`) | **Yes** (Heavy row-by-row logging) | Slowest | No |
| **`TRUNCATE`**| DDL | All Rows (clears whole table) | **No** (Resets table storage directly) | Faster | **Yes** |
| **`DROP`** | DDL | Entire Table (Schema + Data) | **No** (Irreversible) | Instant | N/A (Table is deleted) |

#### Detailed Scope Differences:
1.  **`DELETE FROM table_name WHERE condition;`**
    -   *Use Case*: Removing specific rows (e.g., "Delete only assignments marked as completed").
    -   *Behavior*: Slow for large datasets because it deletes each row individually and logs the deletion for potential rollbacks.
2.  **`TRUNCATE TABLE table_name;`**
    -   *Use Case*: Fast table reset.
    -   *Behavior*: Drops the table and recreates it. It resets the Auto-Increment counter back to 1.
3.  **`DROP TABLE table_name;`**
    -   *Use Case*: Removing obsolete tables.
    -   *Behavior*: Deletes the table definition, data, and constraints entirely from the schema list.


In [ ]:
# Cleanup Playground (Commented out by default for safety).
# Uncomment the specific query block you wish to run.

connection = establish_connection()

if connection and connection.is_connected():
    cursor = connection.cursor()

    try:
        # --- Option 1: DELETE (DML - Selective Row Deletion) ---
        # query = "DELETE FROM assignments WHERE is_completed = TRUE"
        # cursor.execute(query)
        # print("DELETE executed. Selective rows removed (Undo-able).")

        # --- Option 2: TRUNCATE (DDL - Fast Emptying) ---
        # query = "TRUNCATE TABLE assignments"
        # cursor.execute(query)
        # print("TRUNCATE executed. Table emptied and Auto-Increment reset.")

        # --- Option 3: DROP (DDL - Permanent Destruction) ---
        # query = "DROP TABLE assignments"
        # cursor.execute(query)
        # print("DROP executed. Table schema and data deleted entirely.")

        connection.commit()
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Cleanup failed: {err}")
    finally:
        cursor.close()
        connection.close()

In [ ]:
# Execution Caller: View Assignment Summary
connection = establish_connection()

if connection and connection.is_connected():
    view_assignment_summary(connection)
    connection.close()

# Hands-On Lab 1: Personal Assignment Tracker

The objective of this laboratory exercise is to construct a Command-Line Interface (CLI) application for managing university assignments. The application utilizes the functions defined in the preceding sections.

### Architecture

The application is structured into modular components:
1.  **Database Configuration**: Connection parameters.
2.  **CRUD Operations**: Functions for parameterization.
3.  **The Interactive Menu**: A continuous loop for user interaction.

### Initial Database Setup (Prerequisite)

The database and table must be created prior to executing the application. This can be achieved programmatically via the Python Code Cell below.

---

### Verification via GUI Clients (DBeaver / MySQL Workbench)

Students can visually verify the database creation and data integrity using a GUI client:

#### Utilizing DBeaver (Recommended)
1.  **Open DBeaver** and connect to the local MySQL instance.
2.  In the **Database Navigator** (left panel), expand the connection to verify the presence of the `student_tasks` database.
3.  Expand `student_tasks` -> `Tables`. Double-click the `assignments` table.
4.  Navigate to the **Data** tab in the main panel to observe records. Click the **Refresh** button (circular arrows) at the bottom toolbar to fetch live updates from Python executions.

#### Utilizing MySQL Workbench
1.  **Open MySQL Workbench** and connect to the local MySQL instance.
2.  In the **Schemas** tab (left panel), right-click and select **Refresh All**. Determine if `student_tasks` is present.
3.  Expand `student_tasks` -> `Tables`. Right-click `assignments` and select **Select Rows (Limit 1000)**.
4.  Observe the data grid to verify that operations executed in the notebook affect the database state in real-time.


In [ ]:
import os
import mysql.connector


def setup_lab1_database():
    """
    Programmatically creates the database and table for Lab 1.
    """
    try:
        # Connect to server (without database)
        connection = mysql.connector.connect(
            user="student",
            password=os.environ.get("MYSQL_DB_PASSWORD", "dummy_password"),
            host="127.0.0.1",
        )
        cursor = connection.cursor()

        # Create Database
        cursor.execute("CREATE DATABASE IF NOT EXISTS student_tasks")
        print("Database 'student_tasks' verified/created.")

        # Switch context
        cursor.execute("USE student_tasks")

        # Create Table
        create_table_query = """
        CREATE TABLE IF NOT EXISTS assignments (
            assignment_id INT AUTO_INCREMENT PRIMARY KEY,
            title VARCHAR(255) NOT NULL,
            subject VARCHAR(255) NOT NULL,
            due_date DATE,
            is_completed BOOLEAN DEFAULT FALSE
        )
        """
        cursor.execute(create_table_query)
        print("Table 'assignments' verified/created.")

        connection.commit()
    except mysql.connector.Error as err:
        print(f"Setup failed: {err}")
    finally:
        if "cursor" in locals():
            cursor.close()
        if "connection" in locals():
            connection.close()


setup_lab1_database()

In [ ]:
def main_menu():
    """
    Simulates the interactive CLI menu loop.
    Note: In a Jupyter Notebook, a continuous loop may block execution.
    This example is structured for terminal execution or short simulation.
    """
    connection = establish_connection()
    if not connection:
        print("Pre-requisite connection failed. Ensure MySQL is running.")
        return

    run_cycle = True

    while run_cycle:
        print("\n--- University Assignment Tracker ---")
        print("1. Add Assignment")
        print("2. View Pending Assignments")
        print("3. Complete Assignment")
        print("4. Remove Assignment")
        print("5. View Assignments Sorted (Due Date)")
        print("6. View Assignment Summary (By Subject)")
        print("7. Exit")

        # In standard terminal execution: choice = input("Select an option: ")
        # In Jupyter notebook simulation:
        choice = "7"  # Exiting automatically to prevent blocking

        if choice == "1":
            add_assignment(connection, "DBMS Lab 1", "Computer Science", "2026-04-15")
        elif choice == "2":
            view_assignments(connection)
        elif choice == "3":
            complete_assignment(connection, assignment_id=1)
        elif choice == "4":
            remove_assignment(connection, assignment_id=1)
        elif choice == "5":
            view_assignments_sorted(connection)
        elif choice == "6":
            view_assignment_summary(connection)
        elif choice == "7":
            print("Exiting application.")
            run_cycle = False
        else:
            print("Invalid choice.")

    connection.close()
    print("Connection closed.")


# main_menu()

### Running the CLI Application in a Local Terminal

While Jupyter is excellent for lecture demonstrations, standard Command-Line Applications (CLI) are best executed in an operating system terminal.

To facilitate local execution, a standalone script has been prepared: [assignments_tracker.py](file:///usr/local/google/home/anmolsachdeva/adjunct-courses/dbms/glau/bcsc0034/lecture1/lab_cheatsheet/assignments_tracker.py).

#### Procedure for Execution:

1.  **Dependencies**: Ensure that the MySQL Server is running and the `student_tasks` database exists.
2.  **Navigation**: The user navigates to the script directory:
    ```bash
    cd lab_cheatsheet
    ```
3.  **Execution via Terminal**:
    The virtual environment is activated, and the script is executed:

    ```bash
    # Windows
    ..\venv\Scripts\activate
    python assignments_tracker.py

    # Linux / Mac
    source ../venv/bin/activate
    python assignments_tracker.py
    ```

This allows full interactive usage, allowing the input of custom titles and real-time database interaction.


# Topic 2: Database Design Principles: Normalization and De-Normalization

This section transitions from flat-file data models to relational structures. The objective is to eliminate data redundancy and maintain data integrity.

### Understanding Flat Files

A **Flat File** is a single-table data structure (such as a CSV file or spreadsheet) where all data is stored in plain text without structural relationships between entities. 

#### Why Flat Files Were Prevalent Pre-RDBMS:
-   **Simplicity**: Flat files require no complex database management systems (DBMS) or server installations.
-   **Portability**: Plain text files are easily shared and read by any text editor across operating systems.
-   **Low Overhead**: For small-scale, read-only datasets, flat files consume minimal storage and processing power.

#### Where flat files are still used today:
Even in modern deployments, flat files are standard for non-transactional workloads:
-   **Configuration & Infrastructure (IaC)**: Kubernetes YAML manifests, CI/CD pipelines, and project dependencies (`requirements.txt`).
-   **Application Logging**: High-speed appending of system logs (e.g., Nginx access logs) where database locking would slow performance.
-   **Big Data Science (Data Lakes)**: Petabytes of historical archives are stored in optimized columnar flat files (such as Apache Parquet) for Machine Learning pipelines.

---

### The Scenario: A Single-Table University System

Consider an administrative office utilizing a single spreadsheet to manage student enrollments, course details, and instructor information. While intuitive for simple listings, this "flat" structure is inefficient for programmatic manipulation and data integrity.

#### Sample Unnormalized Data Table (Flat and repetitive file format):

| Student ID | Student Name | Course ID | Course Name | Instructor ID | Instructor Name | Instructor Office |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1001 | Student_A | CS_101 | Database_Systems_Intro | INST_01 | Instructor_X | Office_401 |
| 2002 | Student_B | CS_101 | Database_Systems_Intro | INST_01 | Instructor_X | Office_401 |
| 3003 | Student_C | CS_102 | Data_Structures | INST_02 | Instructor_Y | Office_402 |
| 1001 | Student_A | CS_102 | Data_Structures | INST_02 | Instructor_Y | Office_402 |

> Notice the redundancy: `Instructor_X` and `CS_101` details are duplicated. If `Instructor_X` changes their office, multiple rows must be modified.

---

### Identification of Database Anomalies

The utilization of a single, unnormalized table allows information to be squeezed into one giant list, leading to critical anomalies:

#### 1. Insertion Anomaly (Inability to add records)
To introduce a new course `CS_104` (Cloud Computing) taught by `Instructor_W`, the row cannot be inserted without a Student Record or identifier (as the system requires some student to "anchor" the row).

| Student ID | Student Name | Course ID | Course Name | Instructor ID | Instructor Name | Instructor Office |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **`NULL` ⚠️** | **`NULL` ⚠️** | CS_104 | Cloud_Computing | INST_04 | Instructor_W | Office_404 |

---

#### 2. Update Anomaly (Data Inconsistency)
If `Instructor_X` changes their office location from `Office_401` to `Office_999`, every row containing `Instructor_X` must be updated. Failure to update all occurrences results in stale data.

| Student ID | Student Name | Course ID | Course Name | Instructor ID | Instructor Name | Instructor Office |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1001 | Student_A | CS_101 | Database_Systems_Intro | INST_01 | Instructor_X | **`Office_999` ✅** |
| 2002 | Student_B | CS_101 | Database_Systems_Intro | INST_01 | Instructor_X | **`Office_401` ❌ (Stale)** |

---

#### 3. Deletion Anomaly (Loss of Master Data)
If a student record is deleted (due to dropping a course), the fact that the course exists is inadvertently deleted if no other student is enrolled in that specific row.

*Assume a row where Student_D is the only student in CS_103:*
| Student ID | Student Name | Course ID | Course Name | Instructor ID | Instructor Name | Instructor Office |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 4004 | Student_D | CS_103 | Algorithms | INST_03 | Instructor_Z | Office_403 |

Deleting the `Student_D` record results in the loss of `Algorithms` and `Instructor_Z` information entirely from the system.

---

### The Objective: Normalization

Normalization is the systematic process of organizing data into multiple related tables (breaking the giant list into smaller, logical groups) to minimize redundancy and prevent these anomalies.


### Database Keys Primer

Before diving into Normal Forms, it is essential to understand **Keys**—the attributes used to identify rows and establish relationships.

#### 1. Super Key
A set of one or more columns that uniquely identifies a row in a table. It may contain "extra" columns that are not strictly necessary for uniqueness.
-   **Example**: `(StudentID, StudentName)` is a Super Key because `StudentID` alone is unique, making the combination unique too.

#### 2. Candidate Key
A **minimal Super Key**. It uniquely identifies a row using the fewest columns possible. No subset of a Candidate Key can find rows uniquely.
-   **Example**: `StudentID` is a Candidate Key. If a university uses emails as login identifiers, `StudentEmail` is *also* a Candidate Key.

#### 3. Primary Key (PK)
The specific Candidate Key chosen by the database designer to uniquely identify rows in a table. 
-   **Properties**: It cannot contain `NULL` values and must be unique.
-   **Example**: `StudentID`.
-   **Composite Primary Key**: If the Primary Key consists of more than one column, it is called a **Composite Primary Key** (e.g., `(StudentID, CourseID)` in an `Enrollments` table where neither column alone is unique).

#### 4. Foreign Key (FK)
A column (or set of columns) in one table that refers to the Primary Key of another table. It establishes a link between data in two tables.
-   **Example**: `CourseID` in an `Enrollments` table points back to `CourseID` in the `Courses` table.

---

#### Candidate Key vs. Primary Key: The Distinction

To understand the difference between Candidate Keys and the Primary Key:
-   **Candidate Keys are the Candidates** (the pool of eligible options).
-   **The Primary Key is the Elected Representative** (the single option chosen from the pool).
-   **Example**: If a student can be identified uniquely by `StudentID` or by `UniversityEmail`, both reside in the pool of **Candidate Keys**. The database administrator selects `StudentID` to serve as the **Primary Key**. The `UniversityEmail` remains a Candidate Key (often referred to as an Alternate Key).

---

#### Summary of Key Relationships

To remember the hierarchy, understand how keys inherit properties:
-   ✅ **Every Candidate Key is a Super Key** (a Candidate Key is just a Super Key with no redundant columns).
-   ✅ **The Primary Key is a Candidate Key** (the specific one chosen by the designer).
-   ❌ **Not every Super Key is a Candidate Key** (a Super Key might contain "extra" columns that prevent it from being minimal).
-   ❌ **A Super Key cannot be a Primary Key directly** (it must first qualify as a Candidate Key by being minimal).

*Analogy*: 
-   **Super Key**: Any group of people capable of lifting a heavy box (e.g., ten people, even if two are sufficient).
-   **Candidate Key**: The *minimal* number of people required to lift the box (e.g., exactly two). The removal of any person results in failure to lift.
-   **Primary Key**: The specific group of two people *selected* to perform the lifting operation.


### The Normal Forms Hierarchy

The process of normalization is progressive, achieved by satisfying specific rules at each level.

#### First Normal Form (1NF)
**Rule**: Atomicity. Each cell must contain a single, atomic value. No repeating groups or arrays.

**Before (Violates 1NF: Non-atomic array)**:
| StudentID | Courses |
| :--- | :--- |
| 1001 | 'CS_101, CS_102' |

**After (Satisfies 1NF: Atomic rows)**:
| StudentID | Course |
| :--- | :--- |
| 1001 | 'CS_101' |
| 1001 | 'CS_102' |

---

#### Second Normal Form (2NF)
**Rule**: Must be in 1NF and have **No Partial Dependencies**. All non-prime attributes must depend on the *entire* Primary Key (relevant for composite keys). If an attribute depends on only a *part* of a composite key, it must be separated.

**Scenario**: In a table tracking enrollments, the Primary Key is a composite of `(StudentID, CourseID)`. 

**The Problem (Partial Dependency Violation)**: The `StudentName` attribute depends only on the `StudentID`. It has no relation to the `CourseID`. Storing it here duplicates the name for every course the student takes.

**Before (Violates 2NF: Partial Dependency)**:
| StudentID (PK Part 1) | CourseID (PK Part 2) | StudentName (Depends only on StudentID) |
| :--- | :--- | :--- |
| 1001 | 'CS_101' | 'Student_A' |
| 1001 | 'CS_102' | 'Student_A' |

**The Solution**: Split the table so that `StudentName` is stored only once per student in a separate `Students` table, and the enrollment table only tracks the relationship.

**After (Satisfies 2NF: Eliminates Partial Dependency)**:

*Students Table*:
| StudentID | StudentName |
| :--- | :--- |
| 1001 | 'Student_A' |

*Enrollments Table*:
| StudentID | CourseID |
| :--- | :--- |
| 1001 | 'CS_101' |
| 1001 | 'CS_102' |

---

#### Third Normal Form (3NF)
**Rule**: Must be in 2NF and have **No Transitive Dependencies**. A non-prime attribute (non-key column) must not depend on another non-prime attribute. It must only depend on the Primary Key.

**Scenario**: A table tracks `Courses`. The Primary Key is `CourseID`. Non-key columns are `InstructorID` and `InstructorOffice`.

**The Problem (Transitive Dependency Violation)**: The `InstructorOffice` location depends on the `InstructorID`, NOT the `CourseID`. Storing it here means if an Instructor teaches multiple courses, their office location is duplicated. This is a transitive relationship ($CourseID \rightarrow InstructorID \rightarrow InstructorOffice$).

**Before (Violates 3NF: Transitive Dependency)**:
| CourseID (PK) | InstructorID (Non-Key) | InstructorOffice (Depends on InstructorID, not CourseID) |
| :--- | :--- | :--- |
| 'CS_101' | 'INST_01' | 'Office_401' |
| 'CS_102' | 'INST_01' | 'Office_401' |

**The Solution**: Move the `InstructorOffice` to a separate `Instructors` table so that it is stored only once per Instructor.

**After (Satisfies 3NF: Eliminates Transitive Dependency)**:

*Courses Table*:
| CourseID | InstructorID |
| :--- | :--- |
| 'CS_101' | 'INST_01' |
| 'CS_102' | 'INST_01' |

*Instructors Table*:
| InstructorID | InstructorOffice |
| :--- | :--- |
| 'INST_01' | 'Office_401' |

---

#### Boyce-Codd Normal Form (BCNF)
**Rule**: A stricter version of 3NF. For every functional dependency $X \rightarrow Y$ (where column $X$ determines column $Y$), the **determinant $X$** (the column doing the determining) must be a Super Key (Candidate Key).

> **Layman's Terms for $X \rightarrow Y$**: It reads as "$X$ determines $Y$". If the value of $X$ (e.g., `Instructor`) is known, the value of $Y$ (e.g., `CourseName`) can be uniquely found. Here, `Instructor` is the **determinant** (the left side of the equation).

**Scenario**: A table tracks `Students`, `CourseNames`, and `Instructors`. An Instructor teaches only one CourseName. A CourseName can be taught by multiple Instructors. Primary Key is the composite `(StudentID, CourseName)`.

**The Problem (Overlapping Candidate Keys Violation)**: The `Instructor` determines the `CourseName` ($Instructor \rightarrow CourseName$). However, the `Instructor` is a determinant but NOT a Candidate Key (multiple students use the same instructor).

**Before (Violates BCNF: Overlapping candidate keys)**:
| StudentID (PK Part 1) | CourseName (PK Part 2) | Instructor (Determinant, but not Candidate Key) |
| :--- | :--- | :--- |
| 1001 | 'Database_Systems_Intro' | 'Instructor_X' |
| 1002 | 'Database_Systems_Intro' | 'Instructor_X' |
| 1001 | 'Data_Structures' | 'Instructor_Y' |

**The Solution**: Separate the Instructor-CourseName mapping from the Student-Instructor enrollment.

**After (Satisfies BCNF: Determinant is Candidate Key)**:

*Instructor-Course Table*:
| Instructor (PK) | CourseName |
| :--- | :--- |
| 'Instructor_X' | 'Database_Systems_Intro' |
| 'Instructor_Y' | 'Data_Structures' |

*Student-Instructor Table*:
| StudentID | Instructor |
| :--- | :--- |
| 1001 | 'Instructor_X' |
| 1002 | 'Instructor_X' |
| 1001 | 'Instructor_Y' |


#### Fourth Normal Form (4NF)
**Rule**: Must be in BCNF and have **No Multi-valued Dependencies**. Independent many-to-many relationships must not be stored in the same table.

**Scenario**: A student has multiple Hobbies and multiple Contact Numbers. These two entities are independent of each other.

**The Problem (Multi-valued Dependency Violation)**: Storing them together leads to a combinatorial explosion of rows (every hobby must be paired with every contact number to avoid missing data).

**Before (Violates 4NF: Multi-valued Dependency)**:
| StudentID | Hobby | ContactNumber |
| :--- | :--- | :--- |
| 1001 | 'Cricket' | '9999988888' |
| 1001 | 'Cricket' | '7777766666' |
| 1001 | 'Music' | '9999988888' |
| 1001 | 'Music' | '7777766666' |

**The Solution**: Separate the independent lists into distinct tables.

**After (Satisfies 4NF: Decoupled Independent Entities)**:

*Student-Hobbies Table*:
| StudentID | Hobby |
| :--- | :--- |
| 1001 | 'Cricket' |
| 1001 | 'Music' |

*Student-Contacts Table*:
| StudentID | ContactNumber |
| :--- | :--- |
| 1001 | '9999988888' |
| 1001 | '7777766666' |

---

#### Fifth Normal Form (5NF) or Project-Join Normal Form (PJ/NF)
**Rule**: Must be in 4NF and contain no join dependencies that are not implied by candidate keys. A table should be decomposed into smaller tables if joining them back creates the original table without spurious rows.

**Scenario**: A University tracks which Student takes which Course from which Instructor.
-   **Rule**: If Instructor $I$ teaches Course $C$, and Student $S$ takes Course $C$, then Student $S$ is *always* taught by Instructor $I$ (no multiple section choices).

**The Problem (Redundant Join Dependency Violation)**: Storing the 3-way relationship `(StudentID, CourseID, InstructorID)` is redundant if it can be reconstructed by joining binary tables flawlessly.

**Before (Violates 5NF: Join Dependency)**:
| StudentID | CourseID | InstructorID |
| :--- | :--- | :--- |
| 1001 | 'CS_101' | 'INST_01' |
| 1001 | 'CS_102' | 'INST_02' |
| 1002 | 'CS_101' | 'INST_01' |

**The Solution**: Decompose into binary tables. Joining them yields the master list without data loss or spurious rows.

**After (Satisfies 5NF: Decomposed into Binary Joins)**:

*Student-Course Table*:
| StudentID | CourseID |
| :--- | :--- |
| 1001 | 'CS_101' |
| 1001 | 'CS_102' |
| 1002 | 'CS_101' |

*Course-Instructor Table*:
| CourseID | InstructorID |
| :--- | :--- |
| 'CS_101' | 'INST_01' |
| 'CS_102' | 'INST_02' |

---

#### What are Spurious Rows (Phantom Joins)?
**Spurious rows** are "false" or "phantom" rows that appear when two tables are joined together over columns that do not uniquely identify the relationship. These rows contain data combinations that were *never present* in the original unnormalized dataset.

**The Problem**: If tables are decomposed incorrectly (lossy decomposition), joining them back creates "phantom" data.

**Example of a Lossy Join (Creating Spurious Data)**:

*Original Data (What we want to preserve)*:
| StudentID | CourseID | InstructorID |
| :--- | :--- | :--- |
| 1001 | 'CS_101' | 'INST_01' |
| 1002 | 'CS_102' | 'INST_01' |

*Incorrect Decomposition into Binary Tables*:

*Table A (Student-Instructor)*:
| StudentID | InstructorID |
| :--- | :--- |
| 1001 | 'INST_01' |
| 1002 | 'INST_01' |

*Table B (Course-Instructor)*:
| CourseID | InstructorID |
| :--- | :--- |
| 'CS_101' | 'INST_01' |
| 'CS_102' | 'INST_01' |

*Joining Table A and Table B over `InstructorID` (The Spurious Result)*:
| StudentID | CourseID | InstructorID |
| :--- | :--- | :--- |
| 1001 | 'CS_101' | 'INST_01' |
| **1001** | **'CS_102'** | **'INST_01'** | **⚠️ Spurious Row (Falsely suggests Student 1001 takes CS_102)**
| **1002** | **'CS_101'** | **'INST_01'** | **⚠️ Spurious Row (Falsely suggests Student 1002 takes CS_101)**
| 1002 | 'CS_102' | 'INST_01' |

*Key Takeaway*: To prevent spurious rows, tables must be decomposed such that the join attribute is a **Candidate Key** for at least one of the tables (a **Lossless-Join Decomposition**).


#### Sixth Normal Form (6NF)
**Rule**: A table is in 6NF if it satisfies no non-trivial join dependencies (meaning it cannot be decomposed further). In practice, this means tables are decomposed to the point where each table consists of a Primary Key and at most **one** non-key attribute (plus timestamps for temporal data).

**Scenario**: A University tracks temporal data (information that changes over time, like Student Addresses).

**The Problem (Duplicate History Tracking)**: Storing multiple history-tracked attributes in one table results in duplication when only one attribute changes.

**Before (Violates 6NF: Multiple temporal attributes in one table)**:
If Name and Address are stored together with valid time ranges:

| StudentID | StudentName | Address | ValidFrom | ValidTo |
| :--- | :--- | :--- | :--- | :--- |
| 1001 | 'Student_A' | 'Street A' | Jan 2026 | June 2026 |
| 1001 | **'Student_A'** | 'Street B' | June 2026 | Current |

*Notice that the name `'Student_A'` is written twice, even though it never changed. Deleting or updating history requires modifying multiple rows.*

**The Solution**: Decompose the attributes into independent tables, each with its own time-series timeline.

**After (Satisfies 6NF: Pure Attribute Decomposition)**:

*Student-Names History Table*:
| StudentID | StudentName | ValidFrom | ValidTo |
| :--- | :--- | :--- | :--- |
| 1001 | 'Student_A' | Jan 2026 | Current |

*Student-Addresses History Table*:
| StudentID | Address | ValidFrom | ValidTo |
| :--- | :--- | :--- | :--- |
| 1001 | 'Street A' | Jan 2026 | June 2026 |
| 1001 | 'Street B' | June 2026 | Current |

---

> **Is 6NF ALWAYS related to Time-Series?**
> Theoretically, **No**. A table is in 6NF if it contains a Primary Key and at most one non-key attribute (e.g., separating static `(ID, Name, Gender)` into `(ID, Name)` and `(ID, Gender)`). 
> Practically, **Yes**. In standard relational design, splitting static data this far creates unnecessary JOIN overhead. Therefore, 6NF is almost exclusively reserved for **Temporal Databases** (or Data Vault/Anchor Modeling in Data Warehouses), where attributes change at different frequencies over time.


### Normalization vs. De-Normalization Comparison

The following table summarizes the trade-offs between normalized and De-Normalized database models:

| Feature | Normalization | De-Normalization |
| :--- | :--- | :--- |
| **Primary Objective** | Eliminate redundancy, maintain integrity | Optimize read performance, simplify queries |
| **Storage Requirement** | Lower (less duplication) | Higher (intentional duplication) |
| **Write Performance** | Faster (smaller tables, less locking) | Slower (multiple tables to update) |
| **Read Performance** | Slower (requires complex JOINs) | Faster (single table scans) |
| **Integrity Maintenance**| Higher (single source of truth) | Lower (risk of inconsistency) |
| **Typical Use Case** | OLTP (Online Transaction Processing) | OLAP (Analytics) / Reporting |

-   *Key Concept for Evaluation*: Strategic De-Normalization is employed when read operations exceed write operations significantly, such as in data warehousing or reporting dashboards.


# Hands-On Lab 2: University Course Manager

The objective of this laboratory exercise is to refactor a poorly designed, single-table database into a robust, normalized relational structure using Python. Subsequently, strategic De-Normalization will be applied to optimize reporting performance.

### Prerequisites

The simulated dataset `unnormalized_data.csv` must be present in the `assets/` directory.

### Phase 1: Flat Data Import

The following Python script reads the CSV file and imports it into a single MySQL table `flat_university_data`. This facilitates the physical observation of anomalies.


### CSV Dataset Structure

The `unnormalized_data.csv` file contains the following columns simulating a flat, unnormalized spreadsheet:

| Column Name | Description | Example |
| :--- | :--- | :--- |
| `student_id` | Unique identifier for the student | `1001` |
| `student_name` | Name of the student | `Student_A` |
| `course_id` | Identifier for the course | `CS_101` |
| `course_name` | Full name of the course | `Database_Systems_Intro` |
| `instructor_id` | Identifier for the instructor | `INST_01` |
| `instructor_name` | Name of the instructor | `Instructor_X` |
| `instructor_office` | Office location of the instructor | `Office_401` |

-   *Key Concept for Evaluation*: Recognizing anomalies in this flat structure is the primary step towards justification for normalization.


In [ ]:
import csv


def import_flat_data(connection, csv_path):
    """
    Imports unnormalized data from a CSV file into a flat MySQL table.
    """
    cursor = connection.cursor()

    create_table_query = """
    CREATE TABLE IF NOT EXISTS flat_university_data (
        student_id INT,
        student_name VARCHAR(255),
        course_id VARCHAR(50),
        course_name VARCHAR(255),
        instructor_id VARCHAR(50),
        instructor_name VARCHAR(255),
        instructor_office VARCHAR(255)
    )
    """
    cursor.execute(create_table_query)

    try:
        # Context Manager (with open): Ensures automatic closure of the file resource upon block exit.
        with open(csv_path, mode="r", encoding="utf-8") as file:
            csv_reader = csv.reader(file)
            next(csv_reader)  # next() advances the iterator to skip the header row.

            insert_query = """
            INSERT INTO flat_university_data
            (student_id, student_name, course_id, course_name, instructor_id, instructor_name, instructor_office)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            """

            for row in csv_reader:
                if (
                    len(row) == 7
                ):  # Defensive Programming: Validate row length before processing.
                    cursor.execute(
                        insert_query, tuple(row)
                    )  # casting list to tuple for parameterization.

        connection.commit()
        print("Flat data imported successfully.")
    except Exception as e:
        connection.rollback()
        print(f"Failed to import flat data: {e}")
    finally:
        cursor.close()


connection = establish_connection()
import_flat_data(connection, "assets/unnormalized_data.csv")

### Database and Table Hierarchy Mapping (Lab 2)

To maintain structural context, the refactored schema is mapped within the database container.

-   **Active Database Container**: `student_tasks`
-   **Contained Tables (Entities)**: `normalized_students`, `normalized_instructors`, `normalized_courses`, `normalized_enrollments`

---

### Phase 2: Structural Refactoring (Normalization)

The flat table is decomposed into multiple related tables to achieve Third Normal Form (3NF).

#### Target Entity-Relationship Diagram (ERD)


#### Zero-Dependency Mermaid Diagram Rendering

To ensure that the Entity-Relationship Diagram (ERD) renders across various Jupyter environments (JupyterLab, VS Code, or Google Colab) without requiring external PyPI installations, a standard Python utility is utilized. 

The function `render_mermaid` encodes standard Mermaid syntax into Base64 format and fetches a rendered image from the `mermaid.ink` web API.


In [ ]:
import base64
from IPython.display import Image, display


def render_mermaid(graph_code):
    """
    Renders Mermaid diagrams into images using the mermaid.ink web API.
    Zero-dependency solution (requires no pip packages).
    """
    graph_code = graph_code.strip()

    graphbytes = graph_code.encode("utf8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")

    image_url = f"https://mermaid.ink/img/{base64_string}"
    display(Image(url=image_url))

In [ ]:
erd_code = """
erDiagram
    normalized_students {
        int student_id PK
        string student_name
    }
    normalized_instructors {
        string instructor_id PK
        string instructor_name
        string instructor_office
    }
    normalized_courses {
        string course_id PK
        string course_name
        string instructor_id FK
    }
    normalized_enrollments {
        int enrollment_id PK
        int student_id FK
        string course_id FK
    }

    normalized_students ||--o{ normalized_enrollments : "enrolls"
    normalized_courses ||--o{ normalized_enrollments : "contains"
    normalized_instructors ||--o{ normalized_courses : "teaches"
"""
render_mermaid(erd_code)

#### Detailed Schema Definitions and Constraints

##### 1. `normalized_students` Table
| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `student_id` | `INT` | `PRIMARY KEY` | Unique identifier for the student. |
| `student_name` | `VARCHAR(255)` | `NOT NULL` | The name of the student. |

##### 2. `normalized_instructors` Table
| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `instructor_id` | `VARCHAR(50)` | `PRIMARY KEY` | Unique identifier for the instructor. |
| `instructor_name` | `VARCHAR(255)` | `NOT NULL` | The name of the instructor. |
| `instructor_office` | `VARCHAR(255)` | - | Office location. |

##### 3. `normalized_courses` Table
| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `course_id` | `VARCHAR(50)` | `PRIMARY KEY` | Unique identifier for the course. |
| `course_name` | `VARCHAR(255)` | `NOT NULL` | The title of the course. |
| `instructor_id` | `VARCHAR(50)` | `FOREIGN KEY` | References `normalized_instructors(instructor_id)`. |

##### 4. `normalized_enrollments` Table
| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `enrollment_id` | `INT` | `PRIMARY KEY`, `AUTO_INCREMENT`| Unique identifier for the enrollment record. |
| `student_id` | `INT` | `FOREIGN KEY` | References `normalized_students(student_id)`. |
| `course_id` | `VARCHAR(50)` | `FOREIGN KEY` | References `normalized_courses(course_id)`. |


In [ ]:
def create_normalized_tables(connection):
    """
    Creates the normalized database schema (3NF).
    """
    cursor = connection.cursor()

    # Order of Execution is critical: Parent tables (independent) must exist before Child tables (dependent with Foreign Keys) can reference them.
    queries = [
        """
        CREATE TABLE IF NOT EXISTS normalized_students (
            student_id INT PRIMARY KEY, -- Surrogate Primary Key
            student_name VARCHAR(255) NOT NULL
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS normalized_instructors (
            instructor_id VARCHAR(50) PRIMARY KEY,
            instructor_name VARCHAR(255) NOT NULL,
            instructor_office VARCHAR(255)
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS normalized_courses (
            course_id VARCHAR(50) PRIMARY KEY,
            course_name VARCHAR(255) NOT NULL,
            instructor_id VARCHAR(50),
            -- Foreign Key Constraint: Maintains Referential Integrity pointing to Instructors.
            FOREIGN KEY (instructor_id) REFERENCES normalized_instructors(instructor_id)
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS normalized_enrollments (
            enrollment_id INT AUTO_INCREMENT PRIMARY KEY,
            student_id INT,
            course_id VARCHAR(50),
            -- Bridge Table: Resolves Many-to-Many relationships between Students and Courses.
            FOREIGN KEY (student_id) REFERENCES normalized_students(student_id),
            FOREIGN KEY (course_id) REFERENCES normalized_courses(course_id)
        )
        """,
    ]

    try:
        for query in queries:
            # Running multiple DDL queries sequentially.
            cursor.execute(query)
        connection.commit()
        print("Normalized tables created successfully.")
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Failed to create normalized tables: {err}")
    finally:
        cursor.close()


connection = establish_connection()
create_normalized_tables(connection)

In [ ]:
def migrate_data(connection):
    """
    Reads data from the flat table and populates the normalized tables.
    Utilizes sets to ensure uniqueness of entities.
    """
    cursor = connection.cursor()

    try:
        # Fetch all data from flat table
        cursor.execute(
            "SELECT student_id, student_name, course_id, course_name, instructor_id, instructor_name, instructor_office FROM flat_university_data"
        )
        records = cursor.fetchall()

        # Utilizing Python Sets (set()) automatically deduplicates entities (prevents duplicate Primary Key errors).
        students = set()
        instructors = set()
        courses = set()
        enrollments = []

        for row in records:
            students.add((row[0], row[1]))
            instructors.add((row[4], row[5], row[6]))
            courses.add((row[2], row[3], row[4]))
            enrollments.append((row[0], row[2]))

        # Batch Insertion (executemany):
        # How it works: The driver attempts to rewrite the query into a single multi-row INSERT (e.g., INSERT ... VALUES (...), (...)) behind the scenes.
        # Advantages: Sends all parameters in a single network round-trip instead of row-by-row.
        # Slower Alternative: Looping and calling `cursor.execute()` for each row individually (forces query parsing and network latency for each row).
        # Note on Scalability: If the dataset is massive, it may exceed MySQL's `max_allowed_packet` limit. In production, standard drivers handle chunking, or system-level bulk ingestion (LOAD DATA INFILE) is used.
        cursor.executemany(
            "INSERT IGNORE INTO normalized_students (student_id, student_name) VALUES (%s, %s)",
            list(students),
        )

        cursor.executemany(
            "INSERT IGNORE INTO normalized_instructors (instructor_id, instructor_name, instructor_office) VALUES (%s, %s, %s)",
            list(instructors),
        )

        cursor.executemany(
            "INSERT IGNORE INTO normalized_courses (course_id, course_name, instructor_id) VALUES (%s, %s, %s)",
            list(courses),
        )

        cursor.executemany(
            "INSERT IGNORE INTO normalized_enrollments (student_id, course_id) VALUES (%s, %s)",
            enrollments,
        )

        connection.commit()
        print("Data migration completed successfully.")
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Migration failed: {err}")
    finally:
        cursor.close()


connection = establish_connection()
migrate_data(connection)

### Verification of Normalized Schema via GUI Clients (DBeaver / MySQL Workbench)

Students can visually verify the success of the normalization process using a GUI client:

#### Utilizing DBeaver (Recommended)
1.  **View New Tables**: Expand `student_tasks` ➔ `Tables` to see the new `normalized_students`, `normalized_courses`, and other tables.
2.  **Visual Schema (ER Diagram)**: In the Database Navigator, double-click the `student_tasks` database and navigate to the **ER Diagram** tab in the main panel. DBeaver renders relationships automatically based on Foreign Key constraints.

#### Utilizing MySQL Workbench
1.  **View New Tables**: Right-click the Schemas panel and select **Refresh All**. Verify tables appear under `student_tasks`.
2.  **Visual Schema (EER Diagram)**: Click **Database** ➔ **Reverse Engineer** in the top menu. Run the wizard to generate an Entity-Relationship Diagram visually mapping Foreign Keys.


### Phase 3: Reconstructing with JOINs

To view the data in a human-readable format (similar to the original flat view), SQL `JOIN` operations are utilized. This demonstrates how data is combined from multiple normalized tables.

The function `view_student_schedule()` performs an `INNER JOIN` across `normalized_students`, `normalized_enrollments`, and `normalized_courses` tables.


In [ ]:
def view_student_schedule(connection):
    """
    Fetches student schedules by joining normalized tables.
    """
    cursor = connection.cursor()

    # INNER JOINs allow the reconstruction of normalized tables into human-readable views.
    query = """
    SELECT s.student_name, c.course_name, i.instructor_name
    FROM normalized_students s -- 's' is a Table Alias (shorthand) used to avoid typing full table names and resolve ambiguity.
    JOIN normalized_enrollments e ON s.student_id = e.student_id -- Bridge Join
    JOIN normalized_courses c ON e.course_id = c.course_id
    JOIN normalized_instructors i ON c.instructor_id = i.instructor_id
    """

    try:
        cursor.execute(query)
        records = cursor.fetchall()
        print("\n--- Student Schedules (Normalized via JOINs) ---")
        for row in records:
            print(f"Student: {row[0]} | Course: {row[1]} | Instructor: {row[2]}")
    except mysql.connector.Error as err:
        print(f"Failed to fetch schedule: {err}")
    finally:
        cursor.close()


connection = establish_connection()
view_student_schedule(connection)

### Phase 4: De-Normalization for Reporting

To optimize read-heavy operations, such as administrative dashboards, strategic De-Normalization is applied. Instead of running heavy `JOIN` queries for aggregated reports, a pre-calculated table is maintained.

#### The `Course_Summary` Table

This table stores pre-calculated totals, such as the total number of students enrolled per instructor.

-   *Key Concept for Evaluation*: This table is populated periodically (e.g., via batch jobs) to reduce load on the transactional system during peak hours.


In [ ]:
def generate_department_report(connection):
    """
    Simulates the generation of a de-normalized report table.
    Calculates student counts per instructor and stores them in a flat table.
    """
    cursor = connection.cursor()

    create_summary_table = """
    CREATE TABLE IF NOT EXISTS Course_Summary (
        instructor_name VARCHAR(255),
        total_students INT,
        last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
    )
    """
    cursor.execute(create_summary_table)

    # Aggregation via SQL: Pushing compute load to the database server is standard industry practice (as opposed to pulling all rows and grouping compute in Python).
    aggregation_query = """
    SELECT i.instructor_name, COUNT(e.student_id)
    FROM normalized_instructors i
    LEFT JOIN normalized_courses c ON i.instructor_id = c.instructor_id
    LEFT JOIN normalized_enrollments e ON c.course_id = e.course_id
    GROUP BY i.instructor_name
    """

    try:
        cursor.execute(aggregation_query)
        results = cursor.fetchall()

        # TRUNCATE TABLE is faster than DELETE FROM. It resets the table storage rather than deleting row-by-row (no transaction logging per row).
        cursor.execute("TRUNCATE TABLE Course_Summary")

        insert_summary = "INSERT INTO Course_Summary (instructor_name, total_students) VALUES (%s, %s)"
        cursor.executemany(insert_summary, results)

        connection.commit()
        print("De-normalized Department Report generated successfully.")
    except mysql.connector.Error as err:
        connection.rollback()
        print(f"Failed to generate report: {err}")
    finally:
        cursor.close()


connection = establish_connection()
generate_department_report(connection)